# LeetCode #1032: Stream of Characters

https://leetcode.com/problems/stream-of-characters/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(Q \cdot W \cdot L)$ | $O(W \cdot L)$ |
| **Optimal: Reverse Trie ★** | $O(W \cdot L + Q \cdot L)$ | $O(W \cdot L)$ |

---

## Understanding the Methods

### Brute Force
For each query character, re-examine the last $L$ characters of the stream against every word. $Q$ = queries, $W$ = words, $L$ = max word length.

### Optimal: Reverse Trie ★
Insert every word reversed into a Trie. On each `query(letter)`, prepend the letter to an active-suffix set and walk all active states one step in the Trie. If any state reaches a terminal node, a word was found. This avoids re-scanning history for every query.

**Constraints:**
* $1 \leq words.length \leq 2000$
* $1 \leq words[i].length \leq 2000$
* $\sum words[i].length \leq 2 \times 10^4$
* $words[i]$ consists of lowercase English letters
* $1 \leq queries \leq 4 \times 10^4$

## Solutions

### C#

In [ ]:
public class StreamChecker {
    private class TrieNode {
        public TrieNode[] Children = new TrieNode[26];
        public bool IsEnd;
    }

    private TrieNode root = new TrieNode();
    // Track all active trie nodes (suffixes currently being matched)
    private List<TrieNode> activeNodes = new List<TrieNode>();

    public StreamChecker(string[] words) {
        // Insert each word reversed so we can match suffixes forward
        foreach (string word in words) {
            TrieNode node = root;
            for (int i = word.Length - 1; i >= 0; i--) {
                int c = word[i] - 'a';
                if (node.Children[c] == null)
                    node.Children[c] = new TrieNode();
                node = node.Children[c];
            }
            node.IsEnd = true;
        }
    }

    public bool Query(char letter) {
        int idx = letter - 'a';
        List<TrieNode> nextNodes = new List<TrieNode>();

        // Always start a new match attempt from the root for this character
        nextNodes.Add(root);

        // Advance every active suffix-match state by one character
        foreach (TrieNode node in activeNodes) {
            if (node.Children[idx] != null)
                nextNodes.Add(node.Children[idx]);
        }

        activeNodes = new List<TrieNode>();
        bool found = false;
        foreach (TrieNode node in nextNodes) {
            if (node != root) {
                // Keep states that still have a valid Trie path
                activeNodes.Add(node);
            }
            if (node.IsEnd) found = true;
        }
        return found;
    }
}

### Python

In [ ]:
class StreamChecker:
    def __init__(self, words: list[str]):
        # Build a Trie from reversed words so suffixes are matched front-to-back
        self.root = {}
        for word in words:
            node = self.root
            for ch in reversed(word):
                node = node.setdefault(ch, {})
            node['#'] = True  # terminal marker

        self.active = []  # list of current trie nodes (live suffix states)

    def query(self, letter: str) -> bool:
        # Seed a new suffix attempt and advance all active states
        next_active = [self.root]
        for node in self.active:
            if letter in node:
                next_active.append(node[letter])

        self.active = []
        found = False
        for node in next_active:
            if letter in node:
                child = node[letter]
                self.active.append(child)
                if '#' in child:
                    found = True
        return found

### Go

In [ ]:
type TrieNode1032 struct {
    children [26]*TrieNode1032
    isEnd    bool
}

type StreamChecker struct {
    root        *TrieNode1032
    activeNodes []*TrieNode1032
}

func Constructor1032(words []string) StreamChecker {
    root := &TrieNode1032{}
    // Insert each word reversed so suffixes match front-to-back
    for _, word := range words {
        node := root
        for i := len(word) - 1; i >= 0; i-- {
            c := word[i] - 'a'
            if node.children[c] == nil {
                node.children[c] = &TrieNode1032{}
            }
            node = node.children[c]
        }
        node.isEnd = true
    }
    return StreamChecker{root: root}
}

func (sc *StreamChecker) Query(letter byte) bool {
    idx := letter - 'a'
    // Start a fresh match attempt from root, then advance all active states
    nextNodes := []*TrieNode1032{sc.root}
    for _, node := range sc.activeNodes {
        if node.children[idx] != nil {
            nextNodes = append(nextNodes, node.children[idx])
        }
    }
    sc.activeNodes = sc.activeNodes[:0]
    found := false
    for _, node := range nextNodes {
        if node.children[idx] != nil {
            sc.activeNodes = append(sc.activeNodes, node.children[idx])
            if node.children[idx].isEnd {
                found = true
            }
        }
    }
    return found
}

### Rust

In [ ]:
struct TrieNode {
    children: [Option<Box<TrieNode>>; 26],
    is_end: bool,
}

impl TrieNode {
    fn new() -> Self {
        TrieNode { children: Default::default(), is_end: false }
    }
}

struct StreamChecker {
    root: Box<TrieNode>,
    stream: Vec<usize>, // store character indices of the stream
}

impl StreamChecker {
    fn new(words: Vec<String>) -> Self {
        let mut root = Box::new(TrieNode::new());
        // Insert each word reversed so we match suffixes going forward
        for word in &words {
            let mut node = &mut *root;
            for ch in word.chars().rev() {
                let idx = (ch as u8 - b'a') as usize;
                if node.children[idx].is_none() {
                    node.children[idx] = Some(Box::new(TrieNode::new()));
                }
                node = node.children[idx].as_mut().unwrap();
            }
            node.is_end = true;
        }
        StreamChecker { root, stream: Vec::new() }
    }

    fn query(&mut self, letter: char) -> bool {
        // Prepend new character (work backwards through stream checking Trie)
        self.stream.push((letter as u8 - b'a') as usize);
        let mut node = &*self.root;
        // Walk the stream backwards (= forward in reversed Trie)
        for &idx in self.stream.iter().rev() {
            match &node.children[idx] {
                None => return false,
                Some(child) => {
                    if child.is_end { return true; }
                    node = child;
                }
            }
        }
        false
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `words = ["cd","f","kl"]`; queries: `a,b,c,d,e,f`
On query `d`: the reverse Trie has `d→c` (for "cd"). After seeing `c` then `d`, the active state reaches the terminal — returns `true`. On query `f`: `f` is a one-character word, returns `true` immediately.

### 2. Slightly Complex
**Input:** `words = ["abc","bcd"]`; queries: `a,b,c,d`
After `a,b,c` the state for "abc" is active and returns `true`. Then on `d`, the state for "bcd" (`d→c→b`) reaches its terminal — returns `true` again.

### 3. Edge Case: Time Factor
**Input:** $2000$ words each of length $10$; $40{,}000$ queries with no match
Every query advances up to $L = 10$ active states. Total work is $O(Q \cdot L) = 4 \times 10^5$ steps — fast in practice despite the large query count.

### 4. Edge Case: Space Factor
**Input:** $2000$ words, total $\sum len = 2 \times 10^4$ characters
The Trie occupies $O(\sum len) = O(2 \times 10^4)$ nodes in the worst case (no shared prefixes after reversal). The active-state list is bounded by the number of unique Trie nodes at any depth.

### 5. Almost-Impossible but Plausible
**Input:** `words = ["a"]`; queries: all lowercase letters, then `a`
Only the single-character word "a" is in the dictionary. The Trie is one node deep. Every query before `a` returns `false`; the query for `a` immediately hits the terminal and returns `true`.